In [ ]:
!pip install -q torch torchvision timm datasets matplotlib scikit-learn

In [ ]:
import copy
import gc
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.v2 as T
import torchvision.transforms.v2.functional as TF
from datasets import load_dataset
from PIL import Image
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
ENCODERS = {
    "DINOv3": "vit_base_patch16_dinov3.lvd1689m",
    "SigLIP 2": "vit_base_patch16_siglip_224.v2_webli",
    "PE-Core": "vit_pe_core_base_patch16_224.fb",
}
FINETUNE_MODEL_NAME = "vit_small_patch16_dinov3.lvd1689m"
IMAGE_SIZE = 224
GRID = IMAGE_SIZE // 16
NUM_SEG_CLASSES = 150
IGNORE_INDEX = 255
# ADE20K slice for the dense probe, mini-ImageNet slice for the global probe
SEG_TRAIN, SEG_EVAL = 200, 100
CLS_TRAIN_PER_CLASS, CLS_EVAL_PER_CLASS = 10, 5
SEED = 0

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"DEVICE = {DEVICE}")

# Vision Foundation Encoders

Almost every vision system now starts from a frozen pretrained encoder, and the choice of encoder
matters more than the head. This notebook measures what each 2025 encoder family is good at, on
real data and with identical probes. It then rebuilds the training signal behind DINOv3's dense
features (self-distillation plus **Gram anchoring**) and tests it with a short fine-tune.

---

# 1) Introduction

## 1.1 The problem

An encoder maps an image to a global vector **and** a grid of patch vectors. Downstream tasks pull
on different parts:

| Consumer | Reads | Needs |
|---|---|---|
| Zero-shot classification, retrieval | global token | semantic, text-aligned |
| VLM (LLaVA-style) | all patch tokens, projected | semantic patches that a language model can read |
| Segmentation, depth, correspondence | patch tokens | spatially precise, locally consistent patches |

## 1.2 Why one objective rarely gives both

| Family | Objective | Strength | Weakness |
|---|---|---|---|
| Language-supervised (CLIP → SigLIP 2, PE) | Match a **global** image embedding to its caption | Semantic, text-aligned global token | Nothing in the loss asks individual patches to be precise, so patch maps are noisy |
| Self-supervised (DINO → DINOv2 → DINOv3) | Self-distillation on global + masked-patch tokens | Clean, local patch features | No language alignment; global token slightly behind on ImageNet |

DINOv3 found a second problem: when self-supervised training runs long enough to keep improving
the global token, **dense quality degrades**. Patch features drift toward the global token and
lose locality. **Gram anchoring** fixes this.

## 1.3 Findings from the past year

| Model | Date | What is new |
|---|---|---|
| **DINOv3** (Meta) | Aug 2025 | Self-supervised up to **7B** params; **Gram anchoring** regularises the patch-feature Gram matrix towards an earlier checkpoint so dense features stop degrading during long schedules; distilled to ViT-S/B/L/H+ and ConvNeXt. Leads dense tasks: 86.6 mIoU on PASCAL VOC vs 72.7 for SigLIP 2; trails SigLIP 2 (89.1) and PE-Core (89.3) on ImageNet |
| **SigLIP 2** (Google) | Feb 2025 | Sigmoid image–text loss plus captioning decoder, self-distillation and masked prediction; NaFlex variant keeps native aspect ratio. Default encoder in most VLMs; strong global token, noisier patches |
| **Perception Encoder** (Meta) | Apr 2025 | One contrastive recipe; the best features sit in *intermediate* layers, extracted by alignment tuning into PE-Core (global), PE-Lang (for VLMs) and PE-Spatial (dense) |
| **RADIO / C-RADIO** (NVIDIA) | 2024–25 | Agglomerative distillation of several teachers (CLIP, DINO, SAM) into one student |
| **SigLino** (TII) | CVPR 2026 | Distils DINOv3 + SigLIP 2 into small dense (30M) and MoE (0.6B total) encoders |

## 1.4 What this notebook covers

| Section | Content |
|---|---|
| 2) Setup | Three ViT-B/16 encoders at 224 px; ADE20K and mini-ImageNet slices |
| 3) The math | DINO self-distillation loss, Gram matrix and Gram anchoring, the two probes |
| 4) From scratch | k-NN probe, dense linear probe + mIoU, DINO loss, Gram loss, each tested |
| 5) End-to-end | Global vs dense scores for DINOv3 / SigLIP 2 / PE-Core; patch PCA maps |
| 6) Variants | Which layer is best; fine-tuning DINOv3-S with and without Gram anchoring |
| 7) Practical notes | Choosing an encoder, resolution, registers, licenses |
| 8) Conclusion | Decision table |

---

# 2) Setup

## 2.1 Data

- **ADE20K** (`scene_parse_150` validation): scene images with per-pixel labels over 150 classes.
  200 images train a dense probe, and 100 measure mIoU.
- **mini-ImageNet** (validation split, 100 classes): 10 images per class form the k-NN memory
  bank, and 5 per class are queried.

Images are resized on the short side and centre-cropped to 224, giving a $14 \times 14$ patch grid.

In [ ]:
def center_crop(
    img: Image.Image,
    size: int = IMAGE_SIZE,
    interp: TF.InterpolationMode = TF.InterpolationMode.BICUBIC,
) -> torch.Tensor:
    """PIL image -> (C, size, size) uint8 tensor."""
    return TF.pil_to_tensor(
        TF.center_crop(TF.resize(img, size, interpolation=interp), size)
    )


start = time.time()
ade = load_dataset(
    "merve/scene_parse_150",
    data_files={"validation": "data/validation-*.parquet"},
    split="validation",
    verification_mode="no_checks",
)
seg_images = torch.stack(
    [center_crop(ade[i]["image"].convert("RGB")) for i in range(SEG_TRAIN + SEG_EVAL)]
)
# ADE label 0 is "other": shift to 0..149 and mark it ignored
seg_masks = (
    torch.stack(
        [
            center_crop(ade[i]["annotation"], interp=TF.InterpolationMode.NEAREST)[0]
            for i in range(SEG_TRAIN + SEG_EVAL)
        ]
    ).long()
    - 1
)
seg_masks[seg_masks < 0] = IGNORE_INDEX

mini = load_dataset(
    "timm/mini-imagenet",
    data_files={"validation": "data/validation-*.parquet"},
    split="validation",
    verification_mode="no_checks",
)
mini_labels = np.array(mini["label"])
rng = np.random.default_rng(SEED)
train_idx, eval_idx = [], []
for c in range(100):
    idx = rng.permutation(np.where(mini_labels == c)[0])
    train_idx += idx[:CLS_TRAIN_PER_CLASS].tolist()
    eval_idx += idx[
        CLS_TRAIN_PER_CLASS : CLS_TRAIN_PER_CLASS + CLS_EVAL_PER_CLASS
    ].tolist()
cls_images = torch.stack(
    [center_crop(mini[i]["image"].convert("RGB")) for i in train_idx + eval_idx]
)
cls_labels = torch.tensor(mini_labels[train_idx + eval_idx])
print(f"ADE20K: {tuple(seg_images.shape)} images, {tuple(seg_masks.shape)} masks")
print(
    f"mini-ImageNet: {len(train_idx)} memory + "
    f"{len(eval_idx)} query images, 100 classes"
)
print(f"loaded in {time.time() - start:.0f} s")

## 2.2 Encoders

All three are ViT-B/16 (~86–93 M parameters) at 224 px, so differences come from the training
objective, not size. Each uses its own normalisation constants. Each also has its own **prefix
tokens** before the patch tokens: DINOv3 has 1 CLS + 4 registers, SigLIP 2 has none (it pools with
an attention head), and PE-Core has 1 CLS.

In [ ]:
def free_memory() -> None:
    """Release cached accelerator memory after a model is deleted."""
    gc.collect()
    if DEVICE.type == "mps":
        torch.mps.empty_cache()
    elif DEVICE.type == "cuda":
        torch.cuda.empty_cache()


def device_memory_gb() -> float:
    """Memory currently held by the accelerator, in GB."""
    if DEVICE.type == "cuda":
        return torch.cuda.max_memory_allocated() / 1e9
    if DEVICE.type == "mps":
        return torch.mps.driver_allocated_memory() / 1e9
    return 0.0


def load_encoder(name: str) -> nn.Module:
    """timm encoder without classifier; DINOv3 pools by its CLS token, as released."""
    kwargs = {"global_pool": "token"} if "dinov3" in name else {}
    return (
        timm.create_model(
            name, pretrained=True, num_classes=0, img_size=IMAGE_SIZE, **kwargs
        )
        .to(DEVICE)
        .eval()
    )


def normalize(images: torch.Tensor, model: nn.Module) -> torch.Tensor:
    """uint8 (batch_num, 3, H, W) -> float input with the model's own mean/std."""
    mean = torch.tensor(model.pretrained_cfg["mean"], device=images.device).view(
        1, 3, 1, 1
    )
    std = torch.tensor(model.pretrained_cfg["std"], device=images.device).view(
        1, 3, 1, 1
    )
    return (images.float() / 255 - mean) / std


for label, name in ENCODERS.items():
    model = load_encoder(name)
    print(
        f"{label:9s} "
        f"{sum(p.numel() for p in model.parameters()) / 1e6:5.1f} M params | "
        f"prefix tokens {model.num_prefix_tokens} | pooling {model.global_pool} | "
        f"license {model.pretrained_cfg.get('license')}"
    )
    del model
    free_memory()

---

# 3) The math

## 3.1 Self-distillation (DINO)

A **student** $g_{\theta_s}$ and a **teacher** $g_{\theta_t}$ (an exponential moving average of the
student, $\theta_t \leftarrow m\,\theta_t + (1-m)\,\theta_s$) map two augmented views $x_1, x_2$ of
the same image to $K$ logits over learned prototypes. The student matches the teacher's
distribution on the *other* view:

$$
\mathcal{L}_{\text{DINO}} = -\tfrac{1}{2}\big(P_t(x_1)^\top \log P_s(x_2) + P_t(x_2)^\top \log P_s(x_1)\big),
$$

$$
P_s(x) = \operatorname{softmax}\!\big(g_{\theta_s}(x)/\tau_s\big),\qquad
P_t(x) = \operatorname{softmax}\!\big((g_{\theta_t}(x) - \mathbf{c})/\tau_t\big).
$$

Nothing stops the trivial solution where every image gets the same output. Two operations
prevent it. **Centring** subtracts a running mean $\mathbf{c}$ of teacher logits, which prevents
one prototype from dominating. **Sharpening** with $\tau_t \ll \tau_s$ (0.04 vs 0.1) prevents the
uniform distribution. DINOv2/v3 add the same loss on masked patch tokens (iBOT), so patches are
trained too.

## 3.2 The Gram matrix of patch features

Let $P \in \mathbb{R}^{N \times d}$ be the $N$ patch features of one image with rows normalised to
unit length. Their Gram matrix

$$
G = P P^\top \in \mathbb{R}^{N\times N}, \qquad G_{ij} = \cos(p_i, p_j),
$$

records **which patches look alike**. That is exactly what a segmenter relies on, and it is
invariant to any rotation of the feature space ($P \to PQ$ leaves $G$ unchanged).

## 3.3 Gram anchoring

DINOv3 adds, late in training,

$$
\mathcal{L}_{\text{Gram}} = \big\lVert P_s P_s^\top - P_a P_a^\top \big\rVert_F^2,
$$

where $P_a$ comes from an earlier **anchor** checkpoint, chosen while dense features were still
good. Because it constrains only $G$, the student may keep changing its features, for example
making them more semantic, as long as patch-to-patch similarity is preserved. The full objective is
$\mathcal{L}_{\text{DINO}} + \mathcal{L}_{\text{iBOT}} + \lambda\,\mathcal{L}_{\text{Gram}}$.

## 3.4 Probes

**k-NN (global).** Cosine similarity between L2-normalised global features. Each query takes a
similarity-weighted vote among its $k$ nearest memory images, with weights $e^{s/0.07}$. No
training, and it measures how well the raw embedding geometry separates classes.

**Dense linear probe.** One linear layer $W \in \mathbb{R}^{d \times 150}$ on frozen patch features,
logits upsampled bilinearly to pixels, trained with cross-entropy. Score is mean IoU over classes
$c$ that occur in the evaluation set:

$$
\text{mIoU} = \frac{1}{|\mathcal{C}|}\sum_{c \in \mathcal{C}} \frac{|\hat{Y}=c \,\wedge\, Y=c|}{|\hat{Y}=c \,\vee\, Y=c|}.
$$

A linear probe can only recover what is already linearly present in each patch, so it measures
the features rather than the head.

---

# 4) From-scratch implementation

## 4.1 Feature extraction

One pass returns both the global feature (the model's own pooling) and the patch grid with prefix
tokens removed.

In [ ]:
@torch.no_grad()
def extract(
    model: nn.Module, images: torch.Tensor, chunk: int = 50
) -> tuple[torch.Tensor, torch.Tensor]:
    """uint8 (n, 3, 224, 224) -> global (n, embed_dim), patches (n, GRID, GRID, dim).

    Both are returned on CPU.
    """
    globals_, patches = [], []
    for i in range(0, len(images), chunk):
        tokens = model.forward_features(
            normalize(images[i : i + chunk].to(DEVICE), model)
        )
        globals_.append(model.forward_head(tokens, pre_logits=True).cpu())
        patch_tokens = tokens[:, model.num_prefix_tokens :]
        patches.append(patch_tokens.reshape(len(patch_tokens), GRID, GRID, -1).cpu())
    return torch.cat(globals_), torch.cat(patches)


model = load_encoder(ENCODERS["DINOv3"])
g, p = extract(model, seg_images[:4])
print(f"global {tuple(g.shape)} | patches {tuple(p.shape)}  (n, GRID, GRID, embed_dim)")

## 4.2 k-NN probe

In [ ]:
def knn_accuracy(
    feats: torch.Tensor,
    labels: torch.Tensor,
    num_train: int,
    k: int = 20,
    temperature: float = 0.07,
) -> float:
    """Weighted k-NN top-1; first num_train rows are the memory, the rest queries."""
    feats = F.normalize(feats.float(), dim=-1)
    bank, queries = feats[:num_train], feats[num_train:]
    sims, idx = (queries @ bank.T).topk(k, dim=1)
    votes = torch.zeros(len(queries), int(labels.max()) + 1)
    votes.scatter_add_(1, labels[:num_train][idx], (sims / temperature).exp())
    return (votes.argmax(1) == labels[num_train:]).float().mean().item()


num_train = len(train_idx)
start = time.time()
g_cls, _ = extract(model, cls_images)
print(
    "DINOv3 k-NN top-1 on mini-ImageNet: "
    f"{knn_accuracy(g_cls, cls_labels, num_train):.3f} "
    f"(chance 0.010) | features in {time.time() - start:.0f} s"
)

## 4.3 Dense linear probe and mIoU

Patch features are standardised with training-set statistics. The probe trains on random batches
of the 200 training images, with the logits upsampled to 112 px to save memory, and is evaluated
at full 224 px resolution.

In [ ]:
def mean_iou(
    pred: torch.Tensor, target: torch.Tensor, num_classes: int = NUM_SEG_CLASSES
) -> float:
    """pred, target (n, H, W) class maps -> mIoU over classes present in target."""
    valid = target != IGNORE_INDEX
    confusion = torch.bincount(
        target[valid] * num_classes + pred[valid], minlength=num_classes**2
    )
    confusion = confusion.reshape(num_classes, num_classes).double()
    intersection = confusion.diag()
    union = confusion.sum(0) + confusion.sum(1) - intersection
    present = confusion.sum(1) > 0
    return (intersection[present] / union[present]).mean().item()


def dense_probe(
    patches: torch.Tensor, masks: torch.Tensor, steps: int = 300, lr: float = 1e-3
) -> float:
    """Train a linear head on frozen (n, GRID, GRID, embed_dim) patches.

    Returns the mIoU on the held-out images.
    """
    train, evaluate = patches[:SEG_TRAIN], patches[SEG_TRAIN:]
    mu, sd = train.mean((0, 1, 2)), train.std((0, 1, 2)) + 1e-6
    train = ((train - mu) / sd).to(DEVICE)
    train_masks = masks[:SEG_TRAIN].to(DEVICE)
    head = nn.Linear(patches.shape[-1], NUM_SEG_CLASSES).to(DEVICE)
    optimizer = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=1e-4)
    generator = torch.Generator().manual_seed(SEED)
    for _ in range(steps):
        idx = torch.randint(0, SEG_TRAIN, (32,), generator=generator)
        # (32, GRID, GRID, 150) -> (32, 150, 112, 112)
        logits = F.interpolate(
            head(train[idx]).permute(0, 3, 1, 2), size=112, mode="bilinear"
        )
        target = F.interpolate(
            train_masks[idx][:, None].float(), size=112, mode="nearest"
        )[:, 0].long()
        loss = F.cross_entropy(logits, target, ignore_index=IGNORE_INDEX)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    preds = []
    with torch.no_grad():
        # chunks keep the (chunk, 150, 224, 224) upsampled logits small
        for i in range(0, len(evaluate), 20):
            logits = head(((evaluate[i : i + 20] - mu) / sd).to(DEVICE)).permute(
                0, 3, 1, 2
            )
            preds.append(
                F.interpolate(logits, size=IMAGE_SIZE, mode="bilinear").argmax(1).cpu()
            )
    return mean_iou(torch.cat(preds), masks[SEG_TRAIN:])

Check the metric on ground truth (one class clamped away), then probe DINOv3 patches.

In [ ]:
print(
    "mIoU of the ground truth against itself: "
    f"{mean_iou(seg_masks.clamp(max=149), seg_masks):.3f}"
)
start = time.time()
_, p_seg = extract(model, seg_images)
print(
    "DINOv3 dense probe on ADE20K: mIoU "
    f"{dense_probe(p_seg, seg_masks):.3f} | {time.time() - start:.0f} s"
)

## 4.4 DINO loss with centring and sharpening

Tested on real outputs: the frozen backbone with a fresh prototype head, run on two augmentations
of real images. To see what the guards guard against, the teacher logits are then pushed toward
collapse by adding a constant bias to one prototype, the direction a collapsing network drifts
in. Centring subtracts that bias out again. Sharpening is what keeps the centred distribution
away from uniform.

In [ ]:
class DINOHead(nn.Module):
    """MLP -> L2-normalised bottleneck -> weight-normalised prototypes (K logits)."""

    def __init__(
        self, embed_dim: int, num_prototypes: int = 2048, bottleneck: int = 256
    ) -> None:
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, 1024), nn.GELU(), nn.Linear(1024, bottleneck)
        )
        self.prototypes = nn.utils.parametrizations.weight_norm(
            nn.Linear(bottleneck, num_prototypes, bias=False)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x (batch_num, embed_dim) -> logits (batch_num, num_prototypes)."""
        return self.prototypes(F.normalize(self.mlp(x), dim=-1))


def dino_loss(
    student_logits: torch.Tensor,
    teacher_logits: torch.Tensor,
    center: torch.Tensor,
    student_temp: float = 0.1,
    teacher_temp: float = 0.04,
) -> torch.Tensor:
    """Logits (2 * batch_num, K), views stacked [view1; view2] -> scalar loss.

    Cross-view cross-entropy: the student on one view predicts the teacher on the other.
    """
    teacher_probs = F.softmax((teacher_logits - center) / teacher_temp, dim=-1)
    student_logp = F.log_softmax(student_logits / student_temp, dim=-1)
    t1, t2 = teacher_probs.chunk(2)
    s1, s2 = student_logp.chunk(2)
    return -0.5 * ((t1 * s2).sum(-1).mean() + (t2 * s1).sum(-1).mean())


def entropy(probs: torch.Tensor) -> float:
    """Mean entropy (nats) of rows of probs."""
    return -(probs * probs.clamp_min(1e-12).log()).sum(-1).mean().item()


augment = T.Compose(
    [
        T.RandomResizedCrop(IMAGE_SIZE, scale=(0.32, 1.0), antialias=True),
        T.RandomHorizontalFlip(),
        T.RandomApply([T.ColorJitter(0.4, 0.4, 0.2, 0.1)], p=0.8),
        T.RandomGrayscale(0.2),
    ]
)


def two_views(images: torch.Tensor) -> torch.Tensor:
    """uint8 (batch_num, 3, H, W) -> (2 * batch_num, 3, 224, 224).

    Each image gets two independent augmentations.
    """
    return torch.stack(
        [augment(img) for img in images] + [augment(img) for img in images]
    )

Run the guards on real logits: a fresh head on frozen DINOv3, with one prototype biased upward to
mimic a collapsing teacher.

In [ ]:
head = DINOHead(model.embed_dim).to(DEVICE)
batch = cls_images[:64]
views = normalize(two_views(batch).to(DEVICE), model)
with torch.no_grad():
    logits = head(model.forward_head(model.forward_features(views), pre_logits=True))
# a collapsing teacher: every image leans toward prototype 0
collapsed = logits.clone()
collapsed[:, 0] += 0.5
center = collapsed.mean(0)
print(
    f"logit range {logits.min():.2f} .. {logits.max():.2f} "
    f"| log K = {np.log(2048):.2f} nats (uniform)"
)
print("teacher           | entropy (nats) | share of prototype 0")
for label, c, temp in [
    ("centred, T=0.04 ", center, 0.04),
    ("no centring     ", torch.zeros_like(center), 0.04),
    ("centred, T=1    ", center, 1.0),
]:
    probs = F.softmax((collapsed - c) / temp, dim=-1)
    print(f"{label}  | {entropy(probs):14.2f} | {probs[:, 0].mean().item():.3f}")
print(
    f"DINO loss between the two views: {dino_loss(logits, logits, logits.mean(0)):.3f}"
)

## 4.5 Gram loss

Test on real patch features: the Gram loss is ~0 for a rotated copy of the features (it only sees
patch-to-patch similarity) and positive once the features are perturbed.

In [ ]:
def gram(patches: torch.Tensor) -> torch.Tensor:
    """Patches (batch_num, num_patches, embed_dim) or (batch_num, GRID, GRID, dim).

    Returns cosine Gram matrices (batch_num, num_patches, num_patches).
    """
    if patches.dim() == 4:
        patches = patches.flatten(1, 2)
    patches = F.normalize(patches, dim=-1)
    return patches @ patches.transpose(-1, -2)


def gram_loss(
    student_patches: torch.Tensor, anchor_patches: torch.Tensor
) -> torch.Tensor:
    """Squared Frobenius distance between Gram matrices, averaged over the batch."""
    return (gram(student_patches) - gram(anchor_patches)).pow(2).sum((-2, -1)).mean()


feats = p_seg[:8].float()
rotation, _ = torch.linalg.qr(torch.randn(feats.shape[-1], feats.shape[-1]))
print(f"Gram loss vs a rotated copy : {gram_loss(feats @ rotation, feats):.2e}")
print(
    "Gram loss vs a noisy copy   : "
    f"{gram_loss(feats + 0.5 * feats.std() * torch.randn_like(feats), feats):.2e}"
)
print(f"Gram loss vs another image  : {gram_loss(feats.roll(1, dims=0), feats):.2e}")
del model, head
free_memory()

---

# 5) End-to-end: three encoders, two probes

## 5.1 Global vs dense

In [ ]:
results, all_patches = {}, {}
for label, name in ENCODERS.items():
    start = time.time()
    model = load_encoder(name)
    g_cls, _ = extract(model, cls_images)
    _, p_seg = extract(model, seg_images)
    knn = knn_accuracy(g_cls, cls_labels, num_train)
    miou = dense_probe(p_seg, seg_masks)
    results[label] = (knn, miou)
    all_patches[label] = p_seg[SEG_TRAIN : SEG_TRAIN + 4]
    print(
        f"{label:9s} k-NN top-1 {knn:.3f} | ADE20K mIoU {miou:.3f} | "
        f"{time.time() - start:.0f} s | device memory {device_memory_gb():.2f} GB"
    )
    del model
    free_memory()

## 5.2 What the patches look like

Project each encoder's patch features on the evaluation images to their top 3 principal
components and show them as RGB. Clean dense features give regions that follow object
boundaries. Noisy ones give speckle.

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for row in range(4):
    axes[row, 0].imshow(seg_images[SEG_TRAIN + row].permute(1, 2, 0))
    for col, label in enumerate(ENCODERS, start=1):
        feats = all_patches[label][row].reshape(GRID * GRID, -1).numpy()
        rgb = PCA(n_components=3).fit_transform(feats)
        rgb = (rgb - rgb.min(0)) / (rgb.max(0) - rgb.min(0))
        axes[row, col].imshow(rgb.reshape(GRID, GRID, 3), interpolation="nearest")
        if row == 0:
            axes[row, col].set_title(label)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

---

# 6) Variants

## 6.1 Which layer?

The Perception Encoder paper's central claim is that a contrastive encoder's best dense features
sit in **intermediate** layers, and the last layers specialise toward the global
text-alignment objective. The same dense probe on blocks 4, 8 and 12 tests that for each encoder.

In [ ]:
@torch.no_grad()
def extract_layers(
    model: nn.Module, images: torch.Tensor, indices: list[int], chunk: int = 50
) -> list[torch.Tensor]:
    """Patch grids (n, embed_dim, GRID, GRID) after each block in indices, on CPU."""
    outs = [[] for _ in indices]
    for i in range(0, len(images), chunk):
        inter = model.forward_intermediates(
            normalize(images[i : i + chunk].to(DEVICE), model),
            indices=indices,
            intermediates_only=True,
        )
        for j, feats in enumerate(inter):
            outs[j].append(feats.cpu())
    return [torch.cat(o) for o in outs]


layer_idx = [3, 7, 11]
print("encoder   | " + " | ".join(f"block {i + 1:2d}" for i in layer_idx))
for label, name in ENCODERS.items():
    model = load_encoder(name)
    layers = extract_layers(model, seg_images, layer_idx)
    # (n, embed_dim, GRID, GRID) -> (n, GRID, GRID, embed_dim)
    scores = [dense_probe(feats.permute(0, 2, 3, 1), seg_masks) for feats in layers]
    print(f"{label:9s} | " + " | ".join(f"{s:8.3f}" for s in scores))
    del model
    free_memory()

## 6.2 Fine-tuning with and without Gram anchoring

A small reproduction of the problem Gram anchoring solves. DINOv3 ViT-S is fine-tuned on
mini-ImageNet with the **global** DINO loss only, on a fresh prototype head, for 100 steps. Nothing
in that loss constrains patches, so they are free to drift. One run uses the DINO loss alone. The
other adds $\lambda\,\mathcal{L}_{\text{Gram}}$ against the frozen starting model as the anchor.
Both are measured by the dense probe and by k-NN.

Real DINOv3 training runs for over a million steps with iBOT on patches, and the dense features
degrade only late in that schedule. A 100-step run shows how fast an unconstrained global loss
moves the patches, and how the anchor term enters the optimisation.

In [ ]:
def finetune(
    gram_weight: float, steps: int = 100, batch_num: int = 16, lr: float = 3e-4
) -> nn.Module:
    """DINO self-distillation on mini-ImageNet images, plus gram_weight * Gram loss.

    The Gram anchor is the frozen starting model.
    """
    torch.manual_seed(SEED)
    backbone = load_encoder(FINETUNE_MODEL_NAME).train()
    anchor = copy.deepcopy(backbone).eval().requires_grad_(False)
    student = nn.ModuleDict(
        {"backbone": backbone, "head": DINOHead(backbone.embed_dim)}
    ).to(DEVICE)
    teacher = copy.deepcopy(student).eval().requires_grad_(False)
    center = torch.zeros(2048, device=DEVICE)
    optimizer = torch.optim.AdamW(
        [
            {"params": backbone.parameters(), "lr": lr},
            {"params": student["head"].parameters(), "lr": 1e-3},
        ],
        weight_decay=0.04,
    )
    memory = cls_images[:num_train]
    generator = torch.Generator().manual_seed(SEED)

    def forward(
        model: nn.ModuleDict, x: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """x (batch_num, 3, H, W) -> (prototype logits, patch tokens)."""
        tokens = model["backbone"].forward_features(x)
        return (
            model["head"](tokens[:, 0]),
            tokens[:, model["backbone"].num_prefix_tokens :],
        )

    for step in range(steps):
        idx = torch.randint(0, len(memory), (batch_num,), generator=generator)
        x = normalize(two_views(memory[idx]).to(DEVICE), backbone)
        with torch.no_grad():
            teacher_logits, _ = forward(teacher, x)
            anchor_patches = anchor.forward_features(x)[:, anchor.num_prefix_tokens :]
        student_logits, student_patches = forward(student, x)
        loss = dino_loss(student_logits, teacher_logits, center)
        loss = loss + gram_weight * gram_loss(student_patches, anchor_patches)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(student.parameters(), 3.0)
        optimizer.step()
        with torch.no_grad():
            center = 0.9 * center + 0.1 * teacher_logits.mean(0)
            for p_s, p_t in zip(student.parameters(), teacher.parameters()):
                p_t.mul_(0.99).add_(p_s.detach(), alpha=0.01)
    return backbone.eval()

Measure the starting model, then the two fine-tuned runs. Gram drift is the Gram loss between
each run's held-out patch features and the starting model's.

In [ ]:
start = time.time()
base = load_encoder(FINETUNE_MODEL_NAME)
g0, _ = extract(base, cls_images)
_, p0 = extract(base, seg_images)
print(
    f"{'start':22s} | k-NN {knn_accuracy(g0, cls_labels, num_train):.3f} "
    f"| mIoU {dense_probe(p0, seg_masks):.3f}"
)
del base
free_memory()
for label, weight in [("DINO loss only", 0.0), ("DINO + Gram anchoring", 2.0)]:
    tuned = finetune(weight)
    g1, _ = extract(tuned, cls_images)
    _, p1 = extract(tuned, seg_images)
    drift = gram_loss(p1[SEG_TRAIN:].float(), p0[SEG_TRAIN:].float()).item()
    print(
        f"{label:22s} | k-NN {knn_accuracy(g1, cls_labels, num_train):.3f} | "
        f"mIoU {dense_probe(p1, seg_masks):.3f} | Gram drift {drift:.2f}"
    )
    del tuned
    free_memory()
print(f"{time.time() - start:.0f} s | device memory {device_memory_gb():.2f} GB")

**Reading the result.** One hundred steps of the global DINO loss on a fresh head are enough to
move the features: the Gram drift is large, and both probes lose about 3 points from the
starting model (mIoU and k-NN alike). On this budget the anchor does **not** visibly hold the
patches in place: both runs drift by about the same amount, and the mIoU gap between them is
within noise.

That is what the loss predicts. At step 0 the student equals the anchor, so
$\mathcal{L}_{\text{Gram}} = 0$ and its gradient is zero. It only pushes back once drift has
built up, while the DINO gradient is large from the first step. DINOv3 switches the Gram term on
after about a million steps, against an anchor checkpoint whose dense features are already better
than the current student's, so the term has something to restore from the start. Reproducing
that needs a long schedule; here the run shows the problem (fast patch drift under a global loss),
not the cure.

---

# 7) Practical notes

**Picking an encoder.** For a VLM, use a language-aligned encoder (SigLIP 2 or PE-Lang): the
language model needs patches that already carry semantics a text model can read. For segmentation,
depth, tracking or 3D, use DINOv3 (or PE-Spatial). For retrieval and zero-shot classification, use
SigLIP 2 or PE-Core global embeddings. To get both, an agglomerative model (RADIO, SigLino) or two
encoders side by side is the current answer.

**Resolution.** Dense quality keeps improving with resolution for DINOv3 (its paper evaluates at
512–1024 px with a 16 px patch), while CLIP-family patch maps get noisier. Use SigLIP 2's NaFlex
variant when aspect ratio matters, such as documents or screens.

**Registers.** ViTs trained long enough repurpose low-information patches as global "scratch
space", which shows up as high-norm artefact tokens in the patch map. DINOv3 has 4 dedicated
register tokens to absorb this. Drop them, along with CLS, before using patches, as `extract` does.

**Layer choice.** Do not assume the last layer is best for dense tasks, especially for
contrastive encoders (§6.1). Many VLMs take the penultimate layer of SigLIP for the same reason.

**Licenses.** SigLIP 2 and PE are Apache-2.0. DINOv3 weights ship under Meta's DINOv3 license, so
check its terms for commercial use.

**Probe protocol.** These numbers use a 300-image probe at 224 px and are far below published
benchmarks, which train on the full ADE20K train set at 512 px. The **ranking** is what transfers.

---

# 8) Conclusion

| Use case | Encoder | Why |
|---|---|---|
| Segmentation, depth, correspondence, 3D | DINOv3 | Self-distillation on patches + Gram anchoring keeps dense features clean |
| VLM vision tower | SigLIP 2 / PE-Lang | Language-aligned patches |
| Zero-shot classification, retrieval | SigLIP 2 / PE-Core | Strong global, text-aligned embedding |
| Need both in one model | RADIO / SigLino | Distilled from DINO + CLIP-family teachers |
| Long self-supervised training | Add Gram anchoring | Constrains patch similarity structure, not the features themselves |